Primer punto del taller, vamos a importar las herramientas necesarias para que el archivo se ejecute apropiadamente


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 

pd.set_option("display.max_columns", 50) #set_option cambia la configuración interna de pandas para visualizar los resultados. Existen diferentes configuraciones (estos son algunos ejemplos)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print('Inicio de actividad.')

Inicio de actividad.


Iniciamos la actividad punto por punto.
Parte 1 — Exploración de datos (Pandas)
Actividades: 

Cargar la base de datos

Mostrar:
• Primeras filas
• Información general del dataset
• Resumen estadístico

In [3]:
df = pd.read_csv("ventas_sucias_5000.csv")
df.head()


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,"1,326.00",NaN,peru,Efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.00,2,chile,Tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,"1,203.00",9,Colombia,Efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,"1,304.00",3,Perú,TRANSFERENCIA,2024-01-01 03:00:00
4,Luisa,Monitor,426.00,6,chile,Tarjeta,2024-01-01 04:00:00


Aqui ponemos info general del Dataset 

In [4]:
df.info(),df.shape,df['precio'].dtype

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   str    
 1   producto     5000 non-null   str    
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   str    
 4   pais         5000 non-null   str    
 5   metodo_pago  5000 non-null   str    
 6   fecha        5000 non-null   str    
dtypes: float64(1), str(6)
memory usage: 273.6 KB


(None, (5000, 7), dtype('float64'))

Resumen estadistico, vemos valores nulos cantidad de datos que podemos empezar a limpiar

In [5]:
df.describe(include='all') 

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
count,5000,5000,"4,950.00",4950,5000,5000,5000
unique,6,5,NaN,10,7,4,4825
top,Ana,Mouse,NaN,2,chile,TRANSFERENCIA,2024-01-19 00:00:00
freq,870,1069,NaN,573,733,1276,5
mean,NaN,NaN,"5,053.82",NaN,NaN,NaN,NaN
std,NaN,NaN,"63,379.98",NaN,NaN,NaN,NaN
min,NaN,NaN,10.00,NaN,NaN,NaN,NaN
25%,NaN,NaN,531.00,NaN,NaN,NaN,NaN
50%,NaN,NaN,"1,027.00",NaN,NaN,NaN,NaN
75%,NaN,NaN,"1,519.00",NaN,NaN,NaN,NaN


In [6]:
df.duplicated().sum() # Registros repetidos

np.int64(0)

Colunmna cantidad debe tener todos los datos a numeros 

In [7]:
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')

df['cantidad'].dtype

dtype('float64')

Procedemos a corregir el formato de fecha para que este se estandarice


In [8]:
df['fecha'] = pd.to_datetime(df['fecha'])
df['fecha'].dtype

dtype('<M8[us]')

Estandarizar datos 


In [9]:
df['cliente'].unique()

<StringArray>
['Maria', 'Luisa', 'Carlos', 'Juan', 'Pedro', 'Ana']
Length: 6, dtype: str

In [10]:
df['pais'].unique()

<StringArray>
['peru', 'chile', 'Colombia', 'Perú', 'COL', 'Chile', 'col']
Length: 7, dtype: str

In [11]:
df['pais'] = df['pais'].str.lower()

df['pais'] = df['pais'].replace({
    'perú': 'peru',
    'col': 'colombia',
    'COL': 'colombia',
    'Chile': 'chile',
    'Colombia': 'colombia'
})

In [12]:
df['pais'].unique()

<StringArray>
['peru', 'chile', 'colombia']
Length: 3, dtype: str

In [13]:
df['producto'].unique()

<StringArray>
['Monitor', 'Laptop', 'Celular', 'Mouse', 'Teclado']
Length: 5, dtype: str

In [14]:
df['cantidad'].unique()

array([nan,  2.,  9.,  3.,  6.,  7.,  1.,  8.,  5.,  4.])

In [15]:
df['metodo_pago'].unique()

<StringArray>
['Efectivo', 'Tarjeta', 'TRANSFERENCIA', 'transferencia']
Length: 4, dtype: str

In [16]:
df['metodo_pago'] = df['metodo_pago'].str.lower()
df['metodo_pago'].unique()

<StringArray>
['efectivo', 'tarjeta', 'transferencia']
Length: 3, dtype: str

In [17]:
df.isna().sum()

cliente          0
producto         0
precio          50
cantidad       100
pais             0
metodo_pago      0
fecha            0
dtype: int64

In [18]:
df.info()
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   cliente      5000 non-null   str           
 1   producto     5000 non-null   str           
 2   precio       4950 non-null   float64       
 3   cantidad     4900 non-null   float64       
 4   pais         5000 non-null   str           
 5   metodo_pago  5000 non-null   str           
 6   fecha        5000 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2), str(4)
memory usage: 273.6 KB


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
count,5000,5000,"4,950.00","4,900.00",5000,5000,5000
unique,6,5,NaN,NaN,3,3,NaN
top,Ana,Mouse,NaN,NaN,colombia,transferencia,NaN
freq,870,1069,NaN,NaN,2139,2533,NaN
mean,NaN,NaN,"5,053.82",4.95,NaN,NaN,2024-04-14 18:44:59.280000
min,NaN,NaN,10.00,1.00,NaN,NaN,2024-01-01 00:00:00
25%,NaN,NaN,531.00,3.00,NaN,NaN,2024-02-22 06:45:00
50%,NaN,NaN,"1,027.00",5.00,NaN,NaN,2024-04-14 14:30:00
75%,NaN,NaN,"1,519.00",7.00,NaN,NaN,2024-06-05 14:15:00
max,NaN,NaN,"999,999.00",9.00,NaN,NaN,2024-12-04 00:00:00


In [19]:
df[df['producto'] == 'Mouse'][['cliente', 'precio', 'cantidad']]
#df[df['precio'].isna()] #me muestra datos nulos de columna precio 

,cliente,precio,cantidad
8,Carlos,"1,181.00",9.00
9,Luisa,239.00,1.00
15,Maria,"1,639.00",6.00
26,Maria,"1,509.00",4.00
30,Carlos,"1,852.00",9.00
...,...,...,...
4980,Pedro,"1,664.00",6.00
4981,Ana,975.00,7.00
4992,Pedro,498.00,7.00
4996,Carlos,"1,553.00",2.00


In [20]:
df[df['precio'] > 100000]

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
771,Luisa,Teclado,"999,999.00",1.00,colombia,tarjeta,2024-02-02 03:00:00
1123,Ana,Laptop,"999,999.00",2.00,peru,efectivo,2024-02-16 19:00:00
1159,Pedro,Teclado,"999,999.00",2.00,peru,transferencia,2024-02-18 07:00:00
1165,Luisa,Mouse,"999,999.00",3.00,chile,tarjeta,2024-02-18 13:00:00
1188,Pedro,Mouse,"999,999.00",4.00,peru,transferencia,2024-02-19 12:00:00
1195,Ana,Laptop,"999,999.00",8.00,colombia,tarjeta,2024-02-19 19:00:00
1225,Ana,Monitor,"999,999.00",5.00,colombia,transferencia,2024-02-21 01:00:00
1242,Juan,Mouse,"999,999.00",5.00,colombia,transferencia,2024-02-21 18:00:00
1277,Pedro,Celular,"999,999.00",3.00,chile,tarjeta,2024-02-23 05:00:00
1311,Carlos,Teclado,"999,999.00",4.00,colombia,transferencia,2024-02-24 15:00:00


In [21]:
df['precio'] = df['precio'].replace(999999.00, np.nan)
df['precio'] = df['precio'].fillna(df['precio'].median())
df['precio'].describe()

count   5,000.00
mean    1,017.61
std       566.92
min        10.00
25%       537.00
50%     1,023.00
75%     1,508.00
max     1,997.00
Name: precio, dtype: float64

Con esto concluimos una limpieza general de los datos que hasta el momento se ven congruentes, no habian datos duplicados, se estandarizo la columna pais, formato de la fecha, y metodo de pago, vamos a tener en cuenta el valor max (999.999.00 ya que es significativamente superior al promedio) este se analizara en un analisis exploratorio previo 

In [22]:
df[df['cantidad'].isna()]

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,"1,326.00",NaN,peru,efectivo,2024-01-01 00:00:00
22,Maria,Teclado,"1,237.00",NaN,peru,tarjeta,2024-01-01 22:00:00
52,Juan,Teclado,656.00,NaN,colombia,transferencia,2024-01-03 04:00:00
53,Ana,Monitor,"1,805.00",NaN,colombia,transferencia,2024-01-03 00:00:00
126,Pedro,Teclado,"1,858.00",NaN,colombia,tarjeta,2024-01-06 06:00:00
...,...,...,...,...,...,...,...
4846,Juan,Laptop,"1,026.00",NaN,peru,tarjeta,2024-07-20 22:00:00
4873,Maria,Laptop,"1,495.00",NaN,colombia,transferencia,2024-07-22 01:00:00
4914,Pedro,Mouse,"1,702.00",NaN,colombia,tarjeta,2024-07-23 18:00:00
4963,Luisa,Laptop,"1,104.00",NaN,peru,efectivo,2024-07-25 19:00:00


In [23]:
df['cantidad'] = df['cantidad'].fillna(df['cantidad'].median())
df.isna().sum()

cliente        0
producto       0
precio         0
cantidad       0
pais           0
metodo_pago    0
fecha          0
dtype: int64

Aqui ya concluimos que los datos nulos y el Outlier que estaba en columna precios ya fue solucionado, lo que se hizo fue convertirlos en datos nulos y dichos datos nulos se reemplazaron con la media de cada producto.


Siguiente paso sera crear columnas nuevas tales como total de ventas, promedios, ventas mayores etc.


In [24]:
df['total_ventas'] = df['precio'] * df['cantidad']
df_total_ventas= df['total_ventas'].sum(),
print(f'Este es el total de ventas {df_total_ventas}')


Este es el total de ventas (np.float64(25219599.0),)


In [29]:
ventas_totales = df['total_ventas'].sum()
print(f'El total de ventas es: {ventas_totales}')

El total de ventas es: 25219599.0


In [34]:
df["precio_mediana"] = df.groupby(["producto","pais"])["precio"].transform('median')

display(df)

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha,total_ventas,precio_mediana
0,Maria,Monitor,"1,326.00",5.00,peru,efectivo,2024-01-01 00:00:00,"6,630.00","1,050.00"
1,Luisa,Laptop,55.00,2.00,chile,tarjeta,2024-01-01 01:00:00,110.00,990.00
2,Carlos,Monitor,"1,203.00",9.00,colombia,efectivo,2024-01-01 02:00:00,"10,827.00","1,023.00"
3,Luisa,Monitor,"1,304.00",3.00,peru,transferencia,2024-01-01 03:00:00,"3,912.00","1,050.00"
4,Luisa,Monitor,426.00,6.00,chile,tarjeta,2024-01-01 04:00:00,"2,556.00",922.00
...,...,...,...,...,...,...,...,...,...
4995,Juan,Laptop,"1,582.00",8.00,chile,transferencia,2024-07-27 03:00:00,"12,656.00",990.00
4996,Carlos,Mouse,"1,553.00",2.00,peru,tarjeta,2024-07-27 04:00:00,"3,106.00","1,023.00"
4997,Juan,Monitor,"1,776.00",2.00,chile,efectivo,2024-07-27 05:00:00,"3,552.00",922.00
4998,Pedro,Laptop,"1,366.00",8.00,chile,transferencia,2024-07-27 06:00:00,"10,928.00",990.00


In [36]:
df["precio_promedio"] = df.groupby(["producto","pais"])["precio"].transform('mean')

display(df)


,cliente,producto,precio,cantidad,pais,metodo_pago,fecha,total_ventas,precio_mediana,precio_promedio
0,Maria,Monitor,"1,326.00",5.00,peru,efectivo,2024-01-01 00:00:00,"6,630.00","1,050.00","1,035.16"
1,Luisa,Laptop,55.00,2.00,chile,tarjeta,2024-01-01 01:00:00,110.00,990.00,985.86
2,Carlos,Monitor,"1,203.00",9.00,colombia,efectivo,2024-01-01 02:00:00,"10,827.00","1,023.00","1,020.26"
3,Luisa,Monitor,"1,304.00",3.00,peru,transferencia,2024-01-01 03:00:00,"3,912.00","1,050.00","1,035.16"
4,Luisa,Monitor,426.00,6.00,chile,tarjeta,2024-01-01 04:00:00,"2,556.00",922.00,952.23
...,...,...,...,...,...,...,...,...,...,...
4995,Juan,Laptop,"1,582.00",8.00,chile,transferencia,2024-07-27 03:00:00,"12,656.00",990.00,985.86
4996,Carlos,Mouse,"1,553.00",2.00,peru,tarjeta,2024-07-27 04:00:00,"3,106.00","1,023.00",999.54
4997,Juan,Monitor,"1,776.00",2.00,chile,efectivo,2024-07-27 05:00:00,"3,552.00",922.00,952.23
4998,Pedro,Laptop,"1,366.00",8.00,chile,transferencia,2024-07-27 06:00:00,"10,928.00",990.00,985.86


In [38]:
## columna final de precio

df["precio_final"] = df["precio"].fillna(df["precio_mediana"])
display(df)

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha,total_ventas,precio_mediana,precio_promedio,precio_final
0,Maria,Monitor,"1,326.00",5.00,peru,efectivo,2024-01-01 00:00:00,"6,630.00","1,050.00","1,035.16","1,326.00"
1,Luisa,Laptop,55.00,2.00,chile,tarjeta,2024-01-01 01:00:00,110.00,990.00,985.86,55.00
2,Carlos,Monitor,"1,203.00",9.00,colombia,efectivo,2024-01-01 02:00:00,"10,827.00","1,023.00","1,020.26","1,203.00"
3,Luisa,Monitor,"1,304.00",3.00,peru,transferencia,2024-01-01 03:00:00,"3,912.00","1,050.00","1,035.16","1,304.00"
4,Luisa,Monitor,426.00,6.00,chile,tarjeta,2024-01-01 04:00:00,"2,556.00",922.00,952.23,426.00
...,...,...,...,...,...,...,...,...,...,...,...
4995,Juan,Laptop,"1,582.00",8.00,chile,transferencia,2024-07-27 03:00:00,"12,656.00",990.00,985.86,"1,582.00"
4996,Carlos,Mouse,"1,553.00",2.00,peru,tarjeta,2024-07-27 04:00:00,"3,106.00","1,023.00",999.54,"1,553.00"
4997,Juan,Monitor,"1,776.00",2.00,chile,efectivo,2024-07-27 05:00:00,"3,552.00",922.00,952.23,"1,776.00"
4998,Pedro,Laptop,"1,366.00",8.00,chile,transferencia,2024-07-27 06:00:00,"10,928.00",990.00,985.86,"1,366.00"


In [ ]:
df_ventasMin= df['total_ventas'].min()
print(f'Este es el Min de ventas {df_ventasMin}')

Este es el Min de ventas 10.0


In [ ]:
df.groupby('producto')['cantidad'].sum()

producto
Celular   4,803.00
Laptop    4,992.00
Monitor   4,939.00
Mouse     5,322.00
Teclado   4,718.00
Name: cantidad, dtype: float64

In [ ]:
pais_mayor_ventas =df.groupby('pais')['total_venta'].sum()
print(f'El país con mayor ventas es: {pais_mayor_ventas.idxmax()} con un total de ventas de: {pais_mayor_ventas.max()}')

El país con mayor ventas es: colombia con un total de ventas de: 53944722261.0


In [39]:
df.describe()

,precio,cantidad,fecha,total_ventas,precio_mediana,precio_promedio,precio_final
count,"5,000.00","5,000.00",5000,"5,000.00","5,000.00","5,000.00","5,000.00"
mean,"1,017.61",4.95,2024-04-14 18:44:59.280000,"5,043.92","1,016.92","1,017.61","1,017.61"
min,10.00,1.00,2024-01-01 00:00:00,10.00,907.00,947.63,10.00
25%,537.00,3.00,2024-02-22 06:45:00,"1,676.75","1,023.00","1,008.56",537.00
50%,"1,023.00",5.00,2024-04-14 14:30:00,"3,938.00","1,023.00","1,020.26","1,023.00"
75%,"1,508.00",7.00,2024-06-05 14:15:00,"7,535.00","1,039.00","1,035.16","1,508.00"
max,"1,997.00",9.00,2024-12-04 00:00:00,"17,955.00","1,071.50","1,085.07","1,997.00"
std,566.92,2.55,NaN,"4,100.83",42.45,32.16,566.92


Parte 6 — Interpretación de resultados
Analiza y responde:
¿Los resultados obtenidos tienen sentido?
//Al principio hubieron datos poco congruentes pero una vez limpiando lo que a mi parecer estaba mal o necesitaba algun ajuste, tienden a tener algun sentido
¿Detectaste valores sospechosos?
// Hubieron bastantes valores poco logicos (999.999) la cual considere como mejor opcion eliminarlos y rellenarlos con la media de la colunma completa como tal 
¿El promedio representa correctamente los datos?
Efectivamente el promedio muestra correctamente los datos
¿Qué decisiones tomarías si esta fuera información real de negocio? 
Considero muy necesario crear una macro y estandarizar ingresos en cada uno de los puntos considerando que muchos de los cambios fueron simplemente diversidad de formatos, errores en la digitacion de valores tales como fecha, precio y cantidad y tambien pondria una especie de restriccion que obligue a la persona que esta ingresando los datos rectificar y rellenar todo para no dejar celdas vacias.
